# Predictive Sales Engine: OLS Regression, Stepwise Selection, and Dummy Encoding

**Academic project.** Follows `01_eda_hypothesis_testing.ipynb`: the tests showed that shelf location and market matter; this notebook quantifies **how much**, and forecasts unit sales for a specific store profile. `Sales` is measured in thousands of units.

> **Dataset not included (course-provided).** This notebook uses a 400-store retail file supplied for a university course (cleaned as `2nd_assignment_clean.csv` by notebook 01; a version of the Carseats data from *An Introduction to Statistical Learning*). The course file is not redistributed, so the notebook is published **without outputs**. Numbers quoted in the markdown come from the original course run. To run it, place the file in `data/`.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'data'

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

Assignment_2 = pd.read_csv(DATA_DIR / '2nd_assignment_clean.csv')
Assignment_2.head()

## Phase 1: Simple linear regression baseline

Does local income predict the price a store charges? A deliberately weak baseline. Original course run: intercept 119.088, income coefficient -0.048 (an almost flat line).

In [ ]:
model_task1 = LinearRegression()
model_task1.fit(Assignment_2[['Income']], Assignment_2[['Price']])
print(f'Intercept: {model_task1.intercept_[0]:.3f}')
print(f'Coefficient (Income): {model_task1.coef_[0][0]:.3f}')

## Phase 2: Forward stepwise selection

Predictors enter in order of absolute correlation with `Sales` and are kept only if adjusted R-squared improves.

Original course run: correlations Price 0.445, Advertising 0.269, Income 0.151, Population 0.049. Selected model: Sales = 12.1884 - 0.0539 Price + 0.1201 Advertising + 0.0109 Income (R-squared 0.294). Population was rejected.

In [ ]:
cont_predictors = ['Income', 'Advertising', 'Population', 'Price']
outcome = 'Sales'

correlation_with_outcome = (Assignment_2[cont_predictors + [outcome]].corr()[outcome]
                            .drop(outcome).abs().sort_values(ascending=False))
print('Correlation ranking (absolute):')
print(correlation_with_outcome.round(3))

predictor_list, best_predictors = [], []
best_adj_r2, best_model = 0, None
for predictor in correlation_with_outcome.index:
    predictor_list.append(predictor)
    results = sm.OLS(Assignment_2[outcome], Assignment_2[predictor_list].assign(const=1)).fit()
    if results.rsquared_adj > best_adj_r2:
        best_adj_r2, best_model, best_predictors = results.rsquared_adj, results, predictor_list.copy()
    else:
        predictor_list.remove(predictor)

equation = f"Sales = {best_model.params['const']:.4f}"
for pred in best_predictors:
    equation += f" {'+' if best_model.params[pred] >= 0 else '-'} {abs(best_model.params[pred]):.4f}*{pred}"
print(f'\nBest model: {equation}')
print(f'R-squared: {best_model.rsquared:.3f} | Adj. R-squared: {best_adj_r2:.3f}')

## Phase 3: Dummy encoding of shelf location

`ShelveLoc` is one-hot encoded with `drop_first=True`, so **Bad** is the reference level.

Original course run: R-squared 0.625 (up from 0.294), F-statistic 131.5 (p = 1.08e-81), AIC down from 1,834 to 1,584. ShelveLoc_Good +4.8254 and ShelveLoc_Medium +1.8954 (both p < 0.0001): with price, advertising, and income held constant, a Good shelf sells about 4,800 more units than a Bad one, and a Medium shelf about 1,900 more.

In [ ]:
predictors_withCategory = ['Price', 'Advertising', 'Income', 'ShelveLoc']
predictors_dummy = pd.get_dummies(Assignment_2[predictors_withCategory], drop_first=True, dtype=int)

model_dummy = sm.OLS(Assignment_2['Sales'], predictors_dummy.assign(const=1)).fit()
print(model_dummy.summary())
print(f'\nAIC stepwise model: {best_model.aic:.0f} | AIC with shelf dummies: {model_dummy.aic:.0f}')

## Phase 4: Point forecast for a new store

Profile: Price $120, Advertising $11K, Income $73K, Bad shelf. Original course run: 5.71 thousand units.

In [ ]:
new_record = pd.DataFrame({
    'Price': [120],
    'Advertising': [11],
    'Income': [73],
    'ShelveLoc_Good': [0],   # Bad shelf: both dummies are 0
    'ShelveLoc_Medium': [0],
})
new_record = new_record[[c for c in predictors_dummy.columns]]
predicted_value = model_dummy.predict(new_record.assign(const=1))
print(f'Predicted unit sales (thousands): {predicted_value.iloc[0]:.2f}')
print(f"Same store with a Good shelf: {predicted_value.iloc[0] + model_dummy.params['ShelveLoc_Good']:.2f}")

## Takeaways and limitations

- Adding shelf placement raises R-squared from 0.294 to 0.625, which makes it the main operational lever in this dataset.
- Limitation: the model is fit and evaluated on the same 400 stores. Next step: a train/test split or cross-validation, and interaction terms (for example US x Advertising).